# 03 - 代码助手系统

本教程介绍代码检索、生成和审查功能。

## 学习目标
- 掌握代码检索器的使用
- 学会使用代码生成Agent
- 理解代码审查规则

## 1. 环境准备

In [ ]:
import sys
sys.path.append('../src')

from code_retriever import CodeRetriever, CodeDocument, CodeChunker, CodeLanguage
from code_agent import CodeAgent, CodeActionType
from review_agent import ReviewAgent, IssueSeverity

## 2. 代码分块

In [ ]:
# 代码分块器
chunker = CodeChunker()

# 示例代码
python_code = '''
def quicksort(arr):
    if len(arr) <= 1:
        return arr
    pivot = arr[len(arr) // 2]
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    return quicksort(left) + middle + quicksort(right)

def binary_search(arr, target):
    left, right = 0, len(arr) - 1
    while left <= right:
        mid = (left + right) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1
'''

# 检测语言
lang = chunker.detect_language(python_code)
print(f"检测到语言: {lang.value}")

In [ ]:
# 分块
chunks = chunker.chunk(python_code, CodeLanguage.PYTHON)
print(f"分块数量: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"\n块 {i+1}: {chunk.name} ({chunk.block_type.value})")
    print(f"  行数: {chunk.num_lines}")

## 3. 代码检索

In [ ]:
# 创建检索器
retriever = CodeRetriever()

# 添加代码文档
codes = [
    ("def bubble_sort(arr):\n    n = len(arr)\n    for i in range(n):\n        for j in range(0, n-i-1):\n            if arr[j] > arr[j+1]:\n                arr[j], arr[j+1] = arr[j+1], arr[j]\n    return arr", "bubble_sort"),
    ("def merge_sort(arr):\n    if len(arr) > 1:\n        mid = len(arr)//2\n        L = arr[:mid]\n        R = arr[mid:]\n        merge_sort(L)\n        merge_sort(R)\n    return arr", "merge_sort"),
    ("def linear_search(arr, x):\n    for i in range(len(arr)):\n        if arr[i] == x:\n            return i\n    return -1", "linear_search"),
]

for code, name in codes:
    retriever.add_document(CodeDocument(content=code, name=name, language=CodeLanguage.PYTHON))

print(f"代码库文档数: {retriever.num_documents}")

In [ ]:
# 检索
results = retriever.search("排序算法", top_k=2)
print("检索结果:")
for r in results:
    print(f"  [{r.score:.4f}] {r.document.name}")

In [ ]:
# 按名称搜索
results = retriever.search_by_name("sort")
print(f"名称包含'sort'的函数: {[r.document.name for r in results]}")

## 4. 代码生成Agent

In [ ]:
# 创建代码Agent
agent = CodeAgent(retriever=retriever)

# 生成代码
result = agent.generate("实现插入排序算法", language=CodeLanguage.PYTHON)
print("生成的代码:")
print(result.code)

In [ ]:
# 代码补全
partial_code = "def factorial(n):\n    if n <= 1:"
result = agent.complete(partial_code, language=CodeLanguage.PYTHON)
print("补全结果:")
print(result.code)

In [ ]:
# 代码解释
code_to_explain = "lambda x: x**2 if x > 0 else 0"
result = agent.explain(code_to_explain)
print(f"解释: {result.explanation}")

## 5. 代码审查

In [ ]:
# 创建审查器
reviewer = ReviewAgent()

# 有问题的代码
bad_code = '''
import os
from utils import *

def process_data(data):
    try:
        result = eval(data)
        password = "admin123"
        return result
    except:
        pass
'''

result = reviewer.review(bad_code, CodeLanguage.PYTHON)
print(f"审查分数: {result.score}/100")
print(f"通过: {result.passed}")

In [ ]:
# 查看问题详情
print("\n发现的问题:")
for issue in result.issues:
    print(f"  [{issue.severity.value}] L{issue.line}: {issue.message}")
    print(f"    类别: {issue.category.value}")

In [ ]:
# 审查好的代码
good_code = '''
import json
from typing import Dict, Any

def process_data(data: str) -> Dict[str, Any]:
    try:
        result = json.loads(data)
        return result
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON: {e}")
'''

result = reviewer.review(good_code, CodeLanguage.PYTHON)
print(f"审查分数: {result.score}/100")
print(f"问题数: {len(result.issues)}")

## 6. 练习

1. 添加更多代码到检索器
2. 尝试生成不同类型的代码
3. 编写有问题的代码，观察审查结果

In [ ]:
# 练习空间


## 总结

本教程介绍了:
- CodeChunker: 代码分块器
- CodeRetriever: 代码检索器
- CodeAgent: 代码生成/补全/解释
- ReviewAgent: 代码审查

下一步: 学习性能基准测试